# 06 — Final DIKW Analysis

Builds traceable report facts. It refuses to invent Wisdom when required model artifacts are absent.

In [1]:
from pathlib import Path
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")
current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / "config" / "config.yaml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate project root containing config/config.yaml")
with (PROJECT_ROOT / "config" / "config.yaml").open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", OUTPUT / "powerbi", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)
fingerprint_paths = [PROJECT_ROOT / "config" / "config.yaml", project_path("shops")]
for key in ("products", "reviews"):
    fingerprint_paths.extend(sorted(project_path(key).glob("shop_*.json")))
seller_candidate = project_path("seller_metrics")
for candidate in (seller_candidate, seller_candidate.with_suffix(".xlsx")):
    if candidate.exists(): fingerprint_paths.append(candidate)
digest = hashlib.sha256()
for path in fingerprint_paths:
    digest.update(str(path.relative_to(PROJECT_ROOT)).encode("utf-8")); digest.update(path.read_bytes())
INPUT_FINGERPRINT = digest.hexdigest()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_FINGERPRINT:", INPUT_FINGERPRINT)

PROJECT_ROOT: /app/workspace/shopee_dikw
INPUT_FINGERPRINT: ffd74b13d032b8f45851e7c2ee986aa9a766cc1f44f58755688039a237e54cd8


In [2]:
required = {"shops":PROCESSED/"shops_clean.csv","reviews":PROCESSED/"reviews_clustered.csv","profile":OUTPUT/"reports"/"cluster_profile.csv","authenticity":OUTPUT/"reports"/"Shop_Authenticity_Report.csv","classification":OUTPUT/"reports"/"Classification_Result.csv","regression":OUTPUT/"reports"/"Regression_Result.csv"}
missing=[name for name,path in required.items() if not path.exists()]
def finite_or_none(value):
    return float(value) if value is not None and np.isfinite(value) else None
facts = {}
if missing:
    print("FINAL ANALYSIS BLOCKED — missing artifacts:", ", ".join(missing))
    pd.DataFrame([{"ready":False,"missing_artifacts":", ".join(missing)}]).to_csv(OUTPUT/"reports"/"final_analysis_readiness.csv",index=False)
else:
    frames={name:pd.read_csv(path) for name,path in required.items()}
    class_metrics=json.loads((OUTPUT/"reports"/"classification_metrics.json").read_text(encoding="utf-8"))
    reg_metrics=json.loads((OUTPUT/"reports"/"regression_metrics.json").read_text(encoding="utf-8"))
    reviews=frames["reviews"]; shops=frames["shops"]; authenticity=frames["authenticity"]; regression=frames["regression"]
    if "Is_Authentic" not in reviews:
        print("FINAL ANALYSIS BLOCKED — human-confirmed authenticity mapping has not been applied")
    else:
        top_factor=regression.sort_values("Absolute_Standardized_Coefficient",ascending=False).iloc[0]
        class_result=frames["classification"]
        class_recall=class_result.assign(correct=lambda d:d["Actual_Label"]==d["Predicted_Label"]).groupby("Actual_Label")["correct"].mean()
        weakest_class=int(class_recall.idxmin())
        facts={"input_fingerprint":INPUT_FINGERPRINT,"suspicious_review_pct":float((reviews["Is_Authentic"]==0).mean()*100),"lowest_authenticity_shop":authenticity.sort_values("Authentic_Review_Rate").iloc[0]["Shop_ID"],"highest_authenticity_shop":authenticity.sort_values("Authentic_Review_Rate").iloc[-1]["Shop_ID"],"classification_accuracy":class_metrics["accuracy"],"classification_macro_f1":class_metrics["macro_f1"],"classification_weakest_class":weakest_class,"classification_weakest_class_recall":float(class_recall.loc[weakest_class]),"regression_r2":reg_metrics["R2"],"strongest_standardized_factor":top_factor["Feature"],"strongest_standardized_coefficient":top_factor["Standardized_Coefficient"],"rating_sales_correlation":finite_or_none(shops[["Rating_Average","Sales"]].corr().iloc[0,1]) if "Sales" in shops else None,"response_conversion_correlation":finite_or_none(shops[["Chat_Response_Rate_pct","Conversion_Rate_pct"]].corr().iloc[0,1]) if "Conversion_Rate_pct" in shops else None,"interpretation_guardrail":"Clusters indicate suspicious patterns, and regression is association rather than causality."}
        (OUTPUT/"reports"/"wisdom_facts.json").write_text(json.dumps(facts,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
        pd.DataFrame([facts]).to_csv(OUTPUT/"powerbi"/"wisdom_summary.csv",index=False)
        pd.DataFrame([{"ready":True,"missing_artifacts":""}]).to_csv(OUTPUT/"reports"/"final_analysis_readiness.csv",index=False)
        display(pd.DataFrame([facts]))

FINAL ANALYSIS BLOCKED — missing artifacts: classification


In [3]:
if facts:
    lines=["# Computed Wisdom Facts","","> Auto-generated from model outputs. Review wording before publication.",""]
    for key,value in facts.items(): lines.append(f"- **{key}**: {value}")
    (OUTPUT/"reports"/"wisdom_facts.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
    print("Final facts exported for the report and Power BI.")
else:
    print("No final Wisdom facts exported until every required artifact exists.")

No final Wisdom facts exported until every required artifact exists.
